# ProcureSight AI — Unusual Purchase Order Detection

## 1. Business Purpose and Analysis Boundaries

Identify purchase orders with unusual purchasing patterns for human review.

This analysis is separate from compliance prediction:
- Compliance prediction learns from known compliance labels.
- Anomaly detection identifies patterns that differ from reference orders.

An anomaly flag does not establish fraud, non-compliance, or a data error.

We will examine order-time numeric inputs and derived purchasing features.
Supplier identity and compliance labels will not be detector inputs.

We will fit the detector using historical reference orders and assess
later orders separately. This avoids using later orders to define
the historical reference distribution.

Without verified anomaly labels, we cannot claim anomaly-detection accuracy.
We will inspect flagged examples, review workload, and detection stability.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "procurement_prepared.parquet"
)

anomaly_df = pd.read_parquet(DATA_PATH)

print(f"Loaded orders: {len(anomaly_df)}")
print(f"Columns: {len(anomaly_df.columns)}")

display(
    anomaly_df[
        [
            "PO_ID",
            "Order_Date",
            "Item_Category",
            "Quantity",
            "Unit_Price",
            "Negotiated_Price",
        ]
    ].head()
)

Loaded orders: 777
Columns: 16


,PO_ID,Order_Date,Item_Category,Quantity,Unit_Price,Negotiated_Price
0,PO-00001,2023-10-17,Office Supplies,1176,20.13,17.81
1,PO-00002,2022-04-25,Office Supplies,1509,39.32,37.34
2,PO-00003,2022-01-26,MRO,910,95.51,92.26
3,PO-00004,2022-10-09,Packaging,1344,99.85,95.52
4,PO-00005,2022-09-08,Raw Materials,1180,64.07,60.53


## 2. Purchasing Features and Chronological Partitions

Candidate numeric features:
- Quantity: number of units ordered.
- Negotiated_Price: negotiated price per unit.
- Negotiated_Order_Value: quantity multiplied by negotiated price.
- Discount_Percent: percentage reduction from the original unit price.

These describe purchasing patterns available when an order is finalized.
Order value is a derived purchasing amount; the dataset does not establish
its currency or confirm actual payment.

Chronological partitions:
- Reference: orders before January 2023, used to fit the detector.
- Validation: January–June 2023, used to inspect scores and review workload.
- Evaluation: July 2023 onward, reserved for the final detector assessment.

These partitions contain no verified anomaly labels. They support
assessment of behaviour over time, rather than accuracy measurement.

Category context will be considered when interpreting flagged orders,
because legitimate purchasing patterns can differ between categories.

In [2]:
anomaly_df = anomaly_df.copy()

anomaly_df["Order_Date"] = pd.to_datetime(
    anomaly_df["Order_Date"], errors="raise"
)

numeric_columns = [
    "Quantity", "Unit_Price", "Negotiated_Price"
]

numeric_values = anomaly_df[numeric_columns].to_numpy(
    dtype=float
)

assert np.isfinite(numeric_values).all(), (
    "Purchasing inputs contain missing or non-finite values."
)
assert (numeric_values > 0).all(), (
    "Purchasing inputs must be positive."
)
assert anomaly_df["Order_Date"].notna().all()

anomaly_df["Negotiated_Order_Value"] = (
    anomaly_df["Quantity"] * anomaly_df["Negotiated_Price"]
)

anomaly_df["Discount_Percent"] = (
    100
    * (anomaly_df["Unit_Price"] - anomaly_df["Negotiated_Price"])
    / anomaly_df["Unit_Price"]
)

ANOMALY_FEATURES = [
    "Quantity",
    "Negotiated_Price",
    "Negotiated_Order_Value",
    "Discount_Percent",
]

assert np.isfinite(
    anomaly_df[ANOMALY_FEATURES].to_numpy(dtype=float)
).all()

anomaly_df = anomaly_df.sort_values(
    ["Order_Date", "PO_ID"]
).reset_index(drop=True)

reference_orders = anomaly_df.loc[
    anomaly_df["Order_Date"] < "2023-01-01"
].copy()

anomaly_validation_orders = anomaly_df.loc[
    (anomaly_df["Order_Date"] >= "2023-01-01")
    & (anomaly_df["Order_Date"] < "2023-07-01")
].copy()

anomaly_evaluation_orders = anomaly_df.loc[
    anomaly_df["Order_Date"] >= "2023-07-01"
].copy()

partition_summary = pd.DataFrame([
    {
        "Partition": name,
        "Orders": len(frame),
        "First_Order": frame["Order_Date"].min(),
        "Last_Order": frame["Order_Date"].max(),
    }
    for name, frame in [
        ("Reference", reference_orders),
        ("Validation", anomaly_validation_orders),
        ("Evaluation", anomaly_evaluation_orders),
    ]
])

assert partition_summary["Orders"].sum() == len(anomaly_df)

display(partition_summary)
display(reference_orders[ANOMALY_FEATURES].describe().round(2))

,Partition,Orders,First_Order,Last_Order
0,Reference,396,2022-01-01,2022-12-30
1,Validation,176,2023-01-02,2023-06-30
2,Evaluation,205,2023-07-01,2024-01-01


,Quantity,Negotiated_Price,Negotiated_Order_Value,Discount_Percent
count,396.00,396.00,396.00,396.00
mean,1068.97,50.92,54195.27,7.87
std,668.50,25.27,44584.03,4.18
min,51.00,9.27,1230.63,1.03
25%,574.00,28.48,19887.74,3.82
50%,1048.50,49.76,44649.25,7.83
75%,1481.00,71.66,77044.63,11.54
max,5000.00,101.44,368400.00,14.98


## 3. Historical Purchasing Distributions

We visualize only the reference orders before fitting the detector.

Histograms show how frequently different values occur.
Category box plots provide context for differences in purchasing patterns.

Large values and points beyond box-plot whiskers are investigation prompts,
not verified anomalies. We retain these orders in the reference data.

Negotiated order value depends on both quantity and negotiated price,
so these features contain related information.

In [3]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

feature_labels = {
    "Quantity": "Order Quantity",
    "Negotiated_Price": "Negotiated Unit Price",
    "Negotiated_Order_Value": "Negotiated Order Value",
    "Discount_Percent": "Discount (%)",
}

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        feature_labels[feature] for feature in ANOMALY_FEATURES
    ],
    vertical_spacing=0.18,
)

for position, feature in enumerate(ANOMALY_FEATURES):
    row = position // 2 + 1
    col = position % 2 + 1

    fig.add_trace(
        go.Histogram(
            x=reference_orders[feature],
            nbinsx=25,
            marker_color="#2563EB",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(
        title_text=feature_labels[feature], row=row, col=col
    )
    fig.update_yaxes(title_text="Orders", row=row, col=col)

fig.update_layout(
    title="Reference Orders — Purchasing Distributions",
    height=700,
    template="plotly_white",
    bargap=0.08,
)

fig.show()

category_fig = px.box(
    reference_orders,
    x="Item_Category",
    y="Negotiated_Order_Value",
    points="outliers",
    hover_data=["PO_ID", "Quantity", "Negotiated_Price"],
    title="Reference Order Value by Item Category",
    labels={
        "Item_Category": "Item Category",
        "Negotiated_Order_Value": "Negotiated Order Value",
    },
    template="plotly_white",
)

category_fig.update_layout(height=450)
category_fig.show()

### Reference Distribution Findings

Most reference quantities are below approximately 2,000 units,
with a small group near 5,000.

Negotiated order value is right-skewed, with a few large values.
Category distributions provide context but do not establish whether
individual orders are legitimate or problematic.

We retain all reference orders.

## 4. Fit an Isolation Forest

We fit an unsupervised Isolation Forest on the four numeric purchasing
features using the 396 reference orders only.

Supplier identity, category, and compliance labels are excluded from
this initial detector. It therefore detects unusual numeric patterns
across the pooled reference orders, rather than category-specific anomalies.

We retain the original numeric values without logarithmic transformation
or standardization for this initial tree-based approach.

Order value is derived from quantity and price, so the inputs are related
and do not represent four independent sources of information.

We negate sklearn's score_samples output so higher scores indicate
more unusual purchasing patterns.

These scores are not probabilities. We have not yet selected a review
threshold, and we do not use the estimator's default flags.

In [4]:
from sklearn.ensemble import IsolationForest

anomaly_detector = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=1,
)

# Fit only on historical reference orders, without target labels.
anomaly_detector.fit(reference_orders[ANOMALY_FEATURES])

# Negate scores so higher means more unusual.
reference_anomaly_scores = -anomaly_detector.score_samples(
    reference_orders[ANOMALY_FEATURES]
)

validation_anomaly_scores = -anomaly_detector.score_samples(
    anomaly_validation_orders[ANOMALY_FEATURES]
)

score_summary = pd.DataFrame({
    "Reference": pd.Series(reference_anomaly_scores).describe(),
    "Validation": pd.Series(validation_anomaly_scores).describe(),
}).round(4)

print(f"Reference orders used for fitting: {len(reference_orders)}")
print(f"Validation orders scored: {len(anomaly_validation_orders)}")
print("Evaluation partition remains unscored.")
print("Higher scores indicate more unusual numeric patterns.")

display(score_summary)

Reference orders used for fitting: 396
Validation orders scored: 176
Evaluation partition remains unscored.
Higher scores indicate more unusual numeric patterns.


,Reference,Validation
count,396.0000,176.0000
mean,0.4805,0.4899
std,0.0425,0.0480
min,0.4234,0.4267
25%,0.4502,0.4516
50%,0.4735,0.4795
75%,0.4993,0.5176
max,0.7600,0.7415


## 5. Inspect Validation Scores and Unusual Orders

We compare reference and validation score distributions using percentages,
because the partitions contain different numbers of orders.

We inspect the ten highest-scoring validation orders alongside their
purchasing features and category context.

These are ranked examples, not confirmed anomalies.
No review threshold has been selected yet.

The reference scores describe orders used to fit the detector;
validation scores describe later orders excluded from fitting.

In [6]:
score_plot_data = pd.concat(
    [
        pd.DataFrame({
            "Partition": "Reference",
            "Anomaly_Score": reference_anomaly_scores,
        }),
        pd.DataFrame({
            "Partition": "Validation",
            "Anomaly_Score": validation_anomaly_scores,
        }),
    ],
    ignore_index=True,
)

# Shared bins make the two distributions easier to compare.
score_min = score_plot_data["Anomaly_Score"].min()
score_max = score_plot_data["Anomaly_Score"].max()

fig = go.Figure()

for partition, color in [
    ("Reference", "#2563EB"),
    ("Validation", "#F97316"),
]:
    values = score_plot_data.loc[
        score_plot_data["Partition"] == partition,
        "Anomaly_Score",
    ]

    fig.add_trace(go.Histogram(
        x=values,
        name=partition,
        histnorm="percent",
        xbins={
            "start": score_min,
            "end": score_max + 0.000001,
            "size": (score_max - score_min) / 25,
        },
        marker_color=color,
        opacity=0.65,
    ))

fig.update_layout(
    title="Anomaly Scores — Reference versus Validation",
    xaxis_title="Anomaly Score (higher = more unusual)",
    yaxis_title="Orders within each partition (%)",
    barmode="overlay",
    template="plotly_white",
    height=450,
)

fig.show()

validation_ranked_orders = anomaly_validation_orders.copy()
validation_ranked_orders["Anomaly_Score"] = (
    validation_anomaly_scores
)

validation_ranked_orders = validation_ranked_orders.sort_values(
    ["Anomaly_Score", "PO_ID"],
    ascending=[False, True],
)

display(
    validation_ranked_orders[
        ["PO_ID", "Order_Date", "Item_Category"]
        + ANOMALY_FEATURES
        + ["Anomaly_Score"]
    ].head(10).round({
    feature: 4
    for feature in ANOMALY_FEATURES + ["Anomaly_Score"]
})
)

,PO_ID,Order_Date,Item_Category,Quantity,Negotiated_Price,Negotiated_Order_Value,Discount_Percent,Anomaly_Score
463,PO-00013,2023-03-09,Electronics,5000,74.86,374300.00,13.6961,0.7415
536,PO-00696,2023-05-13,MRO,1915,102.66,196593.90,4.5112,0.6274
523,PO-00253,2023-04-30,Packaging,1892,96.37,182332.04,9.2561,0.6104
439,PO-00105,2023-02-15,Office Supplies,1936,92.24,178576.64,1.5161,0.6054
530,PO-00682,2023-05-04,Raw Materials,1981,90.99,180251.19,11.1686,0.6032
405,PO-00381,2023-01-14,Packaging,1698,98.73,167643.54,2.2185,0.6026
406,PO-00439,2023-01-14,MRO,1682,101.30,170386.60,4.1627,0.5994
541,PO-00132,2023-05-21,Office Supplies,114,101.54,11575.56,1.9221,0.5981
548,PO-00412,2023-06-02,Packaging,1969,87.28,171854.32,2.2073,0.5800
527,PO-00594,2023-05-02,Office Supplies,166,94.48,15683.68,1.2232,0.5738


## 6. Reference-Based Threshold Comparison

We compare thresholds calculated from the reference anomaly scores:
the 90th, 95th, and 97.5th percentiles.

For example, the 95th percentile marks the upper approximately 5%
of reference scores. Ties can affect the exact flagged proportion.

We apply each historical cutoff unchanged to validation orders.
Later review workload may differ from the reference proportion.

These candidate policies are illustrative and are being examined after
validation score exploration. They are not confirmed business requirements.

Without verified anomaly labels, we cannot select thresholds using
precision, recall, or anomaly-detection accuracy.

In [7]:
threshold_comparison_rows = []

for percentile in [90, 95, 97.5]:
    threshold = float(
        np.percentile(reference_anomaly_scores, percentile)
    )

    reference_flags = reference_anomaly_scores >= threshold
    validation_flags = validation_anomaly_scores >= threshold

    threshold_comparison_rows.append({
        "Reference_Percentile": percentile,
        "Score_Threshold": threshold,
        "Reference_Flagged": int(reference_flags.sum()),
        "Reference_Workload_%": 100 * reference_flags.mean(),
        "Validation_Flagged": int(validation_flags.sum()),
        "Validation_Workload_%": 100 * validation_flags.mean(),
    })

anomaly_threshold_comparison = pd.DataFrame(
    threshold_comparison_rows
)

display(anomaly_threshold_comparison.round(4))

,Reference_Percentile,Score_Threshold,Reference_Flagged,Reference_Workload_%,Validation_Flagged,Validation_Workload_%
0,90.0,0.5295,40,10.1010,32,18.1818
1,95.0,0.5562,20,5.0505,14,7.9545
2,97.5,0.5680,10,2.5253,10,5.6818


## 7. Frozen Anomaly Review Policy

We select the reference-score 95th percentile as the prototype cutoff.

This flags 20 of 396 reference orders (5.05%) and
14 of 176 validation orders (7.95%).

The choice reflects an illustrative review-workload preference after
inspecting validation results. It is not an accuracy-optimal threshold
or a confirmed business requirement.

We freeze:
- The Isolation Forest fitted on 396 reference orders.
- The four numeric purchasing features.
- The full-precision reference 95th-percentile threshold.
- The rule: flag scores greater than or equal to the threshold.

The cutoff remains fixed when assessing later orders.
It does not guarantee a 5% review workload on future data.

Flags identify unusual pooled purchasing patterns for human review.
They do not establish fraud or non-compliance.

In [8]:
FROZEN_ANOMALY_MODEL = anomaly_detector
FROZEN_ANOMALY_FEATURES = ANOMALY_FEATURES.copy()
FROZEN_ANOMALY_PERCENTILE = 95

# Preserve full precision; do not copy the rounded table value.
FROZEN_ANOMALY_THRESHOLD = float(
    np.percentile(
        reference_anomaly_scores,
        FROZEN_ANOMALY_PERCENTILE,
    )
)

print("Frozen detector: Isolation Forest")
print(f"Features: {FROZEN_ANOMALY_FEATURES}")
print(f"Reference percentile: {FROZEN_ANOMALY_PERCENTILE}")
print(f"Full-precision threshold: {FROZEN_ANOMALY_THRESHOLD:.12f}")
print("Evaluation orders remain unscored.")

Frozen detector: Isolation Forest
Features: ['Quantity', 'Negotiated_Price', 'Negotiated_Order_Value', 'Discount_Percent']
Reference percentile: 95
Full-precision threshold: 0.556156317762
Evaluation orders remain unscored.
